In [3]:
import pandas as pd
#importing datset to be used

df = pd.read_csv("/content/IMDB Dataset.csv")

### Importing the Pandas Library

It provides the DataFrame data structure, which makes it easy to load, organize, filter, and analyze tabular data stored in CSV files.

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### Loading the Dataset

The IMDb dataset is loaded from a CSV file into a Pandas DataFrame called `df`. This DataFrame stores all movie reviews and their corresponding sentiment labels (positive or negative), allowing us to perform probability calculations.

In [ ]:
# Display all information about the dataset we are using
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


### Exploring the Dataset

The `head()` function displays the first five rows of the dataset. This allows us to verify that the dataset was loaded correctly and to identify the available columns, namely `review` and `sentiment`.

In [ ]:
# Count the number of positive and negative reviews which are in datset
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
# Count positive reviews in dataset
positive_reviews = (df["sentiment"] == "positive").sum()

# Count total reviews
total_reviews = len(df)

# Calculate the prior probability
prior = positive_reviews / total_reviews

print("Positive Reviews:", positive_reviews)
print("Total Reviews:", total_reviews)
print("Prior Probability P(Positive):", prior)

Positive Reviews: 25000
Total Reviews: 50000
Prior Probability P(Positive): 0.5


### Counting Review Sentiments

The `value_counts()` function counts the number of positive and negative reviews in the dataset. These counts are used to compute the Prior Probability, which represents the probability that a randomly selected review is positive.

In [ ]:
# Select only positive reviews
positive_df = df[df["sentiment"] == "positive"]

# Display the first few positive reviews
positive_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive


In [ ]:
print(positive_df.shape)

(25000, 2)


In [ ]:
keyword = "excellent"

positive_keyword_count = positive_df["review"].str.contains(
    keyword,
    case=False,
    na=False
).sum()

print("Positive reviews containing", keyword, ":", positive_keyword_count)

Positive reviews containing excellent : 2936


### Filtering Positive Reviews

This step creates a new DataFrame containing only positive reviews. Restricting the analysis to positive reviews allows us to calculate the likelihood of a keyword appearing in positive reviews.

In [ ]:
likelihood = positive_keyword_count / positive_reviews

print("Likelihood P(excellent | Positive):", likelihood)

Likelihood P(excellent | Positive): 0.11744


### Calculating the Likelihood

The likelihood measures how often the keyword "excellent" appears among positive reviews. It is calculated by dividing the number of positive reviews containing the keyword by the total number of positive reviews.

In [ ]:
keyword = "excellent"

keyword_count = df["review"].str.contains(
    keyword,
    case=False,
    na=False
).sum()

print("Total reviews containing", keyword, ":", keyword_count)

Total reviews containing excellent : 3625


In [ ]:
marginal = keyword_count / total_reviews

print("Marginal Probability P(excellent):", marginal)

Marginal Probability P(excellent): 0.0725


### Calculating the Marginal Probability

The marginal probability represents the probability that the keyword appears in any review, regardless of whether the review is positive or negative. It is calculated using the entire dataset.

In [ ]:
posterior = (likelihood * prior) / marginal

print("Posterior Probability P(Positive | excellent):", posterior)

Posterior Probability P(Positive | excellent): 0.8099310344827587


### Applying Bayes' Theorem

Bayes' Theorem combines the prior probability, likelihood, and marginal probability to compute the posterior probability. This result indicates the probability that a review is positive given that it contains the keyword "excellent".

In [ ]:


# List of positive keywords
keywords = ["excellent", "brilliant", "wonderful", "perfect"]

# Prior Probability
positive_reviews = (df["sentiment"] == "positive").sum()
total_reviews = len(df)
prior = positive_reviews / total_reviews

# Create a DataFrame containing only positive reviews
positive_df = df[df["sentiment"] == "positive"]


def bayes_probability(keyword):
    """
    Computes Prior, Likelihood, Marginal and Posterior
    for a given keyword.
    """

    # Number of positive reviews containing the keyword
    positive_keyword_count = positive_df["review"].str.contains(
        keyword,
        case=False,
        na=False
    ).sum()

    # Likelihood P(keyword | Positive)
    likelihood = positive_keyword_count / positive_reviews

    # Number of ALL reviews containing the keyword
    keyword_count = df["review"].str.contains(
        keyword,
        case=False,
        na=False
    ).sum()

    # Marginal P(keyword)
    marginal = keyword_count / total_reviews

    # Posterior P(Positive | keyword)
    posterior = (likelihood * prior) / marginal if marginal != 0 else 0

    return {
        "Keyword": keyword,
        "Prior": round(prior, 4),
        "Likelihood": round(likelihood, 4),
        "Marginal": round(marginal, 4),
        "Posterior": round(posterior, 4)
    }


# Calculate probabilities for all keywords
results = []

for keyword in keywords:
    results.append(bayes_probability(keyword))

# Display results as a table
results_df = pd.DataFrame(results)

results_df

,Keyword,Prior,Likelihood,Marginal,Posterior
0,excellent,0.5,0.1174,0.0725,0.8099
1,brilliant,0.5,0.0755,0.0489,0.7711
2,wonderful,0.5,0.1066,0.0650,0.8203
3,perfect,0.5,0.1204,0.0794,0.7579


### Generalizing the Bayesian Calculation

Instead of repeating the same calculations for every keyword, a reusable function is created. The function accepts any keyword as input and computes the Prior Probability, Likelihood, Marginal Probability, and Posterior Probability. This approach follows the DRY (Don't Repeat Yourself) principle by reducing code duplication and improving maintainability.

### Results

The final table summarizes the Bayesian probability calculations for each selected positive keyword. It displays the Prior Probability, Likelihood, Marginal Probability, and Posterior Probability, allowing us to compare how strongly each keyword indicates positive sentiment.